# 버그 찾기 4 (보통~어려움): 모델은 정답 국가를 기대하지 않는다?

위치 힌트: **`continuation_logprob` 함수 안**에 있다. 그 아래의 점검 셀은 전부 통과한다.

모델: Llama 대신 `toy_lm.py`의 장난감 언어모델을 쓴다(CPU에서 몇 초). 이 파일에는 버그가 없고, 읽어도 된다. 데이터는 진짜 `cities.csv`다.

배경: 02 노트북 Cell 06처럼, 국가 이름의 log-prob(EX, 모델이 그 문장을 얼마나 기대하나)만으로 참/거짓이 갈리는지 본다. 갈리면 probe의 1.0을 A5(그럴듯함)로도 설명할 수 있다.

**하는 법**
1. 위에서부터 끝까지 실행하고, 맨 아래 **결론**을 읽는다.
2. 코드나 설명 중 "말과 실제 동작이 다른 곳"을 찾아 **문장으로** 적는다(맨 아래 `내 답` 셀).
3. 고치면 숫자가 어떻게 바뀔지 **한 줄 예측**을 먼저 적는다.
4. 고쳐서 다시 돌린다. 문법이 막히면 고칠 내용을 문장으로 쓰고 `???`로 남겨 두면 Claude가 채운다.

이 노트북은 오류 없이 끝까지 돈다. 정답은 Claude에게 따로 있다.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

from toy_lm import load_toy, log_softmax

cities, tok, model = load_toy()
print(len(cities), "문장, vocab", len(tok.vocab))

In [ ]:
def continuation_logprob(model, tok, prefixes, continuations, batch_size=64):
    """log P(continuation | prefix)의 token 합. prefix 뒤에 continuation을 붙인 전체 문장을 한 번 forward한다."""
    out = []
    for s in range(0, len(prefixes), batch_size):
        P, C = prefixes[s : s + batch_size], continuations[s : s + batch_size]
        enc = tok([p + c for p, c in zip(P, C)], padding=True)
        n_prefix = [len(tok(p)["input_ids"]) for p in P]  # <bos> 포함 prefix 토큰 수
        logp = log_softmax(model(enc["input_ids"]))  # [B, T, V], 각 위치의 log 확률
        for b in range(len(P)):
            L = int(enc["attention_mask"][b].sum())  # pad를 뺀 실제 길이
            target = enc["input_ids"][b, n_prefix[b] : L]  # continuation 토큰들
            picked = np.take_along_axis(logp[b, n_prefix[b] : L], target[:, None], axis=-1)
            out.append(picked.sum())  # log 확률의 합 = 확률 곱의 log
    return np.array(out)


prefixes = [f"The city of {c} is in" for c in cities["city"]]
conts = [f" {c}." for c in cities["country"]]
assert all(p + c == s for p, c, s in zip(prefixes, conts, cities["statement"]))
country_lp = continuation_logprob(model, tok, prefixes, conts)

In [ ]:
# 점검 1: target이 정말 국가 토큰인가
i = 1
enc = tok([prefixes[i] + conts[i]], padding=True)
n = len(tok(prefixes[i])["input_ids"])
L = int(enc["attention_mask"][0].sum())
print("문장:", cities["statement"][i])
print("target 토큰:", tok.decode(enc["input_ids"][0, n:L]))

# 점검 2: log 확률이니 전부 0 이하여야 한다
assert (country_lp <= 0).all()
# 점검 3: 문장마다 값이 하나씩
assert len(country_lp) == len(cities)
print("점검 통과")

In [ ]:
y = cities["label"].to_numpy()
print(f"국가 log-prob만으로 참/거짓 AUROC: {roc_auc_score(y, country_lp):.3f}")
print(f"평균 log-prob  참: {country_lp[y == 1].mean():.2f}   거짓: {country_lp[y == 0].mean():.2f}")

## 결론

1. 국가 log-prob만으로는 참/거짓 AUROC가 0.49다. 참 문장과 거짓 문장의 평균 log-prob도 거의 같다(−8.4 vs −8.6).
2. 즉 이 모델은 정답 국가를 오답 국가보다 더 기대하지 않는다.
3. 따라서 probe의 1.0은 EX(그럴듯함, A5)로 설명되지 않는다. A5는 기각한다.

## 내 답

- 말과 실제 동작이 다른 곳:
- 그래서 틀린 결론 문장(번호):
- 고치면 어떻게 바뀔지 예측:
- 고친 뒤 실제 결과: